In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
import torch

### Part#1: Using Pretrained Models with Pipeline API

#### Sentiment Analysis

In [ ]:
senti_pipe = pipeline("sentiment-analysis")

result = senti_pipe("I love learning Artificial Intelligence!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9996731281280518}]


### Text Generation —> Single Output

In [ ]:
gen = pipeline("text-generation", model="gpt2")

output = gen("Artificial Intelligence will", max_new_tokens=30)
print(output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Artificial Intelligence will be used to help develop a new type of AI for social networks.\n\nIn the past, AI has been used as a tool to help humans'}]


### Text Generation —> Multiple Outputs

In [ ]:
gen = pipeline("text-generation", model="gpt2")

output = gen("Artificial Intelligence will", max_new_tokens=30, num_return_sequences=5)
print(output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Artificial Intelligence will not just be able to take over the world, it will also be able to replace human beings in the process.\n\nBut the question is what'}, {'generated_text': 'Artificial Intelligence will be able to do more than just create a computer that can actually learn.\n\nAs it turns out, AI is going to become so powerful that'}, {'generated_text': 'Artificial Intelligence will now be able to use the ability to connect to the nearest part of your world.\n\nThe ability now has an ability to automatically select a random'}, {'generated_text': "Artificial Intelligence will be a big deal. If you're a software engineer, you will be able to use AI to create algorithms that will be smarter with your data than"}, {'generated_text': 'Artificial Intelligence will continue to evolve. This is because it will be able to take a large amount of data, and be able to see and identify it quickly and accurately'}]


### Part#2: AutoTokenizer and AutoModel

#### AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

raw_in = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

inputs = tokenizer(raw_in, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


#### AutoModel

In [ ]:
model = AutoModel.from_pretrained("distilbert-base-uncased")

outs = model(**inputs)

print(outs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 16, 768])


### Part#3: Model Head —> AutoModelForSequenceClassification

In [ ]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint);

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

ins = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
outs = model(**ins)

print(outs.logits.shape)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

torch.Size([2, 2])


### Part#4: Postprocessing the Output

In [ ]:
print("Logits:")
print(outs.logits)

predicts = torch.nn.functional.softmax(outs.logits, dim=-1)
print("\nProbabilities:")
print(predicts)

print("\nLabel Mapping:")
print(model.config.id2label)

Logits:
tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)

Probabilities:
tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)

Label Mapping:
{0: 'NEGATIVE', 1: 'POSITIVE'}


### Q1: Question Answering Pipeline

In [ ]:
qa_pipeline = pipeline("question-answering")

context = (
    "Artificial Intelligence is a field of computer science that focuses on creating "
    "systems capable of performing tasks that normally require human intelligence. "
    "These tasks include learning, reasoning, and problem solving."
)

ques = "What does Artificial Intelligence focus on?"
res = qa_pipeline(question=ques, context=context)

print("=== Original Question ===")
print("Question :", ques)
print("Answer   :", res["answer"])
print("Score    :", res["score"])
print()
print("What does score represent?")
print("Score is the model's confidence (0 to 1) that the extracted answer is correct.")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

=== Original Question ===
Question : What does Artificial Intelligence focus on?
Answer   : creating systems capable of performing tasks that normally require human intelligence
Score    : 0.6208829879760742

What does score represent?
Score is the model's confidence (0 to 1) that the extracted answer is correct.


In [ ]:
modified_ques = "What tasks does Artificial Intelligence include?"

res_modified = qa_pipeline(question=modified_ques, context=context)

print("=== Modified Question ===")
print("Question :", modified_ques)
print("Answer   :", res_modified["answer"])
print("Score    :", res_modified["score"])

=== Modified Question ===
Question : What tasks does Artificial Intelligence include?
Answer   : learning, reasoning, and problem solving
Score    : 0.9496279358863831


### Q2: Topic Classification using AutoTokenizer and AutoModelForSequenceClassification


In [ ]:
model_name = "cardiffnlp/twitter-roberta-base-topic-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentence = "The latest smartphone features a foldable screen and improved AI camera."
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad(): outputs = model(**inputs)

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
prob_list = probabilities[0].tolist()

print("=== All Class Probabilities ===")
for idx, prob in enumerate(prob_list):
    label = model.config.id2label[idx]
    print(f"  [{idx}] {label}: {prob:.4f}")

sorted_indices = sorted(range(len(prob_list)), key=lambda i: prob_list[i], reverse=True)

second_idx  = sorted_indices[1]
second_prob = prob_list[second_idx]
second_label = model.config.id2label[second_idx]

print("\n=== 2nd Most Likely Class ===")
print(f"Index       : {second_idx}")
print(f"Label       : {second_label}")
print(f"Probability : {second_prob:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== All Class Probabilities ===
  [0] arts_&_culture: 0.0016
  [1] business_&_entrepreneurs: 0.0146
  [2] celebrity_&_pop_culture: 0.0029
  [3] diaries_&_daily_life: 0.0064
  [4] family: 0.0011
  [5] fashion_&_style: 0.0013
  [6] film_tv_&_video: 0.0060
  [7] fitness_&_health: 0.0035
  [8] food_&_dining: 0.0007
  [9] gaming: 0.0112
  [10] learning_&_educational: 0.0049
  [11] music: 0.0043
  [12] news_&_social_concern: 0.0125
  [13] other_hobbies: 0.0085
  [14] relationships: 0.0013
  [15] science_&_technology: 0.9110
  [16] sports: 0.0048
  [17] travel_&_adventure: 0.0017
  [18] youth_&_student_life: 0.0017

=== 2nd Most Likely Class ===
Index       : 1
Label       : business_&_entrepreneurs
Probability : 0.0146


### Q3: Text Summarization using Auto Classes

In [ ]:
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
Artificial Intelligence (AI) has rapidly transformed many areas of modern life,
including healthcare, finance, transportation, and education. In healthcare, AI
algorithms can analyze medical images to detect diseases earlier and more
accurately than humans in some cases. In finance, AI helps detect fraudulent
transactions and optimize investment strategies. In transportation,
autonomous vehicles and smart traffic management systems rely heavily on AI
for efficiency and safety. In education, AI-driven tools can personalize
learning for students and provide intelligent tutoring support. As AI
technologies continue to advance, ethical considerations such as data privacy,
bias, and transparency have become increasingly important for developers and
policymakers.
"""

ins = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

summary_ids = model.generate(
    ins["input_ids"],
    max_new_tokens=80,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("=== Summary ===")
print(summary)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

=== Summary ===
Artificial Intelligence (AI) has rapidly transformed many areas of modern life, including healthcare, finance, transportation, and education. As AI technologies continue to advance, ethical considerations such as data privacy and transparency have become increasingly important.
